In [ ]:
from option_analyzer import *
import random
self = OptionAnalyzer('quotes', 'chain')
pd.set_option("display.max_columns", None)

In [ ]:
def check_option_csv_files():
    csv_files = glob(os.path.expanduser(f'~/lab/data/both~*~*.csv'))
    csv_file_dict = dict([((lambda x: (pd.to_datetime(x[-1][:-4].replace('_', ' ')), x[-2]))(csv.split('~')), csv) for csv in csv_files])
    print(len(csv_files), 'csv files,', len(csv_file_dict), 'server ts pairs')
    ds2sizes = {}
    for dt, svr in csv_file_dict:
        date = dt.strftime('%F')
        key = (svr, date)
        if key not in ds2sizes:
            ds2sizes[key] = []
        csv_file = csv_file_dict[(dt, svr)]
        ds2sizes[key].append(os.path.getsize(csv_file))
    lod = []
    for svr, date in ds2sizes:
        for size in ds2sizes[(svr, date)]:
            lod.append({'server': svr, 'date': date, 'size': size})
    return pd.DataFrame(lod)

def load_intraday_option_data(symbol_regex, dt):
    csv_files = glob(os.path.expanduser(f'~/lab/data/both~*~*.csv'))
    csv_file_dict = dict([((lambda x: (pd.to_datetime(x[-1][:-4].replace('_', ' ')), x[-2]))(csv.split('~')), csv) for csv in csv_files])
    lodf = []
    _date = pd.to_datetime(dt).date() if dt else pd.Timestamp.today().date()
    for key in sorted(csv_file_dict):
        if key[0].date() == _date:
            _df = pd.read_csv(csv_file_dict[key]).drop(columns=['moneyness', 'overpaid', 'leverage'])
            _df = _df[_df.symbol.str.contains(symbol_regex)].copy()
            _df['ts'] = key[0]
            lodf.append(_df)
    return pd.concat(lodf)

def plot_metric_for_strikes_with_max_span(_dfe, col, W=2000, H=500):
    _g = _dfe.groupby(['type', 'symbol', 'strike'])[col]
    _dfg = pd.DataFrame(_g.max() - _g.min()).reset_index().rename(columns={col: col + '_span'})
    _dfm = _dfg.groupby(['type', 'symbol'])[col + '_span'].idxmax()
    _dfs = _dfg.loc[_dfm].merge(_dfe, how='inner', on=['type', 'symbol', 'strike'])
    _dfx = _dfg.loc[_dfm]
    type2strike = dict(zip(_dfx['type'], _dfx['strike']))
    print('OI:', _dfs.loc[:, ['type', 'strike', 'OpenInterest']].drop_duplicates())
    px.scatter(_dfs, x='ts', y=col, color='type', width=W, height=H, title=col + ' ' + str(type2strike)).show()
    px.scatter(_dfs, x='ts', y='ImpVola', color='type', width=W, height=H, title='Implied Volatility ' + str(type2strike)).show()
    return _dfx

In [ ]:
_dfsize = check_option_csv_files()

In [ ]:
_g = _dfsize.groupby(['server', 'date'])
_g.count().join(_g.mean(), rsuffix='_mean').join(_g.std(), rsuffix='_std')

In [ ]:
df = load_intraday_option_data('QQQ|GLD', '2026-09-28')
df.shape

In [ ]:
df.groupby(['symbol', 'type']).expDt.nunique().reset_index()

In [ ]:
for _sym in df.symbol.unique():
    px.bar(df[df.symbol==_sym].groupby(['type', 'expDt']).strike.nunique().reset_index(), x='expDt', y='strike', title=_sym, color='type', barmode='group').show()

### Verify that intraday open interests are static
Min and max open interest changes should both be zero.

In [ ]:
_g = df.groupby(['symbol', 'type', 'strike', 'expDt'])
_dfoi = pd.DataFrame({'ts_min': _g.ts.min(), 'ts_max': _g.ts.max(), 'oi_min': _g.OpenInterest.min(), 'oi_max': _g.OpenInterest.max()}).reset_index()
_dfoi['time_span'] = _dfoi.ts_max - _dfoi.ts_min
_dfoi['oi_change'] = _dfoi.oi_max - _dfoi.oi_min
print(_dfoi.oi_change.min(), _dfoi.oi_change.max())

### Plot gamma values

In [ ]:
_width = 1600
metric = 'Gamma'
for symbol in df.symbol.unique():
    _df = df[df.symbol == symbol]
    random_exp_dt = random.sample(list(_df.expDt.unique()), 1)[0]
    _dfe = _df[(_df.expDt==random_exp_dt)]
    _dte = _dfe.dte.iloc[0]
    px.scatter(_dfe, x='strike', y='OpenInterest', color='type', height=300, width=_width, log_y=True, title=f'Open Interests for {symbol} {_dte} DTE ({random_exp_dt})').show()
    px.scatter(_dfe, x='strike', y=metric, color='type', height=500, width=_width, title=f'{metric} range for {symbol} {_dte} DTE ({random_exp_dt})').show()
    px.scatter(_dfe.loc[:, ['ts', 'lastPrice']].drop_duplicates(), x='ts', y='lastPrice', width=2000, height=300, title=f'{symbol} Spot Price').show()
    plot_metric_for_strikes_with_max_span(_dfe, metric)